# Part 6 — Uncertainty: Bootstrap Confidence Intervals

*Evals from First Principles, Part 6.*

A single score is a point estimate, nothing more. Report **accuracy = 0.70** on 30 graded outputs and it sounds solid, but that 0.70 came from one particular sample of 30 items. Grade a different 30 items and you would not get exactly 0.70 again. The **bootstrap** turns that one labeled set into a **confidence interval**: it resamples the set with replacement many times, recomputes the metric on each resample, and reads the 2.5th and 97.5th percentiles as a 95% CI. We do it by hand on a fixed 30-item correct/incorrect set, print a few resample draws so the mechanic is visible, then show the interval is wide at n=30 and narrows as the set grows.

Pure Python + NumPy, offline, deterministic (fixed seed).


## The labeled set

We start from one eval run's grades: 30 outputs, each marked `1` (correct) or `0` (incorrect). This is the **sample**. The true accuracy of the system is unknown; the sample only gives us an **estimate** of it.


In [ ]:
import numpy as np

# --- The labeled set ------------------------------------------------------
# 30 graded outputs from one eval run: 1 = correct, 0 = incorrect.
# This is the SAMPLE. The true accuracy is unknown; 0.70 is only our estimate.
DATA = np.array([
    1, 1, 0, 1, 1, 1, 0, 1, 1, 0,
    1, 1, 1, 0, 1, 1, 0, 1, 1, 1,
    0, 1, 1, 0, 0, 1, 1, 0, 1, 1,
])


## Step 1 — The bootstrap mechanic

The bootstrap's one move is: resample the same n items, with replacement, and recompute the metric. "With replacement" means each of the n draws is independent and uniform over the original items, so some items get picked twice (or three times) and others get skipped entirely in that resample. Do this many times and the spread of the resulting metric values approximates how much the point estimate would wobble if we had graded a different sample from the same population.

`bootstrap` below does exactly that: draw `n` indices from `0..n-1` with replacement, take the mean of the resampled labels (that is accuracy, since labels are 0/1), and repeat `B` times. It also optionally keeps the first few raw index draws so we can inspect the mechanic directly.


In [ ]:
def bootstrap(data, B, seed, keep=0):
    """Resample `data` with replacement B times; return the metric on each.

    Each replicate draws n indices uniformly from 0..n-1 (WITH replacement),
    so some items appear twice and some not at all. The metric is the mean
    (accuracy) of the resampled labels. Returns the B replicate values and,
    if keep > 0, the first `keep` index draws so the reader can inspect them.
    """
    rng = np.random.default_rng(seed)
    n = len(data)
    reps = np.empty(B)
    draws = []
    for b in range(B):
        idx = rng.integers(0, n, size=n)   # n draws, with replacement
        if b < keep:
            draws.append(idx)
        reps[b] = data[idx].mean()
    return reps, draws


## Step 2 — Reading the interval off the replicates

Once we have `B` replicate accuracy values (one per resample), the 95% bootstrap confidence interval is just the middle 95% of that distribution: the 2.5th percentile as the lower bound and the 97.5th percentile as the upper bound. `ci95` does that lookup.


In [ ]:
def ci95(reps):
    """Read the 2.5th and 97.5th percentiles: the 95% bootstrap interval."""
    lo = np.percentile(reps, 2.5)
    hi = np.percentile(reps, 97.5)
    return lo, hi


## Step 3 — The point estimate

Before bootstrapping anything, print the sample itself and its plain point estimate: count how many of the 30 items are correct and divide by 30.


In [ ]:
line = "=" * 72
dash = "-" * 72
print(line)
print("PART 6 - UNCERTAINTY: the bootstrap turns one score into an interval.")
print(line)

n = len(DATA)
correct = int(DATA.sum())
point = DATA.mean()
print("\nThe labeled set (30 items, 1 = correct, 0 = incorrect):")
print("  " + " ".join(str(x) for x in DATA))
print(f"  correct: {correct} / {n}")
print(f"  point estimate:  accuracy = {correct}/{n} = {point:.4f}")


## Step 4 — Three example resamples

Before running the full B = 10000 bootstrap, look at three individual resamples with the fixed seed (`numpy.random.default_rng(0)`). Each one is a full list of 30 indices drawn with replacement from `0..29`; printing the raw indices makes the "with replacement" mechanic concrete, since you can see the same index appear more than once inside a single draw.


In [ ]:
print("\n" + dash)
print("THE MECHANIC: resample 30 items WITH REPLACEMENT, recompute accuracy.")
print("Three example resamples (fixed seed, numpy default_rng(0)):")
print(dash)
B = 10000
reps, draws = bootstrap(DATA, B, seed=0, keep=3)
for i, idx in enumerate(draws, start=1):
    resampled = DATA[idx]
    c = int(resampled.sum())
    print(f"  draw {i}: indices = {idx.tolist()}")
    print(f"          correct in resample: {c} / {n}  ->  accuracy = {c / n:.4f}")
print("  (repeated indices are the point: 'with replacement' reshuffles weight.)")


## Step 5 — The full bootstrap and the 95% CI

Now use all `B = 10000` replicates (the same `reps` array computed above) and read the 95% CI with `ci95`. The mean of the replicates should land close to the point estimate; the interval around it is the honest uncertainty on that point estimate.


In [ ]:
print("\n" + dash)
print(f"FULL BOOTSTRAP: B = {B} resamples, then read the percentiles.")
print(dash)
lo, hi = ci95(reps)
print(f"  bootstrap replicates    = {B}")
print(f"  mean of replicates      = {reps.mean():.4f}")
print(f"  2.5th percentile        = {lo:.4f}")
print(f"  97.5th percentile       = {hi:.4f}")
print(f"  95% CI                  = [{lo:.4f}, {hi:.4f}]  (width {hi - lo:.4f})")
print(f"  -> report: accuracy = {point:.2f}, 95% CI [{lo:.2f}, {hi:.2f}]")
print("     the point estimate alone hides how much this set could wobble.")


## Step 6 — The interval narrows as n grows

The 21/30 = 0.70 accuracy ratio is a property of the sample proportion. Tile the same 30-item set `k` times (k = 1, 4, 16) to get n = 30, 120, 480 items, all with the identical 0.70 accuracy, and bootstrap each. If uncertainty comes from sample size, not from the accuracy value itself, the interval should shrink as n grows, roughly following the `1/sqrt(n)` rule: quadruple n and the width should roughly halve.


In [ ]:
print("\n" + dash)
print("WIDE AT n=30, NARROWER AS n GROWS (same 0.70 accuracy, bigger set):")
print(dash)
print(f"  {'n':>5}   {'95% CI':<20}  {'width':>7}")
prev_width = None
for k in (1, 4, 16):
    big = np.tile(DATA, k)          # same 21/30 ratio, k times as many items
    reps_k, _ = bootstrap(big, B, seed=0)
    lo_k, hi_k = ci95(reps_k)
    w = hi_k - lo_k
    ratio = "" if prev_width is None else f"  ({prev_width / w:.2f}x tighter)"
    print(f"  {len(big):>5}   [{lo_k:.4f}, {hi_k:.4f}]  {w:>7.4f}{ratio}")
    prev_width = w
print("  -> width roughly halves each time n quadruples (the ~1/sqrt(n) rule).")
print("     n=30 gives an interval too wide to trust a 2-point difference.")
print(line)


## Recap

- A reported score is a **point estimate**, not the truth. It carries uncertainty that depends on how many items it was computed from.
- The **bootstrap** turns one labeled set into a confidence interval by resampling it with replacement `B` times and recomputing the metric on each resample.
- The 95% CI is read directly off the replicate distribution: the 2.5th and 97.5th percentiles.
- At n = 30, accuracy 0.70 carries a 95% CI of `[0.53, 0.87]`, too wide to trust a small difference between two systems.
- The interval narrows as n grows, roughly following the `1/sqrt(n)` rule: quadrupling n roughly halves the width (n=120 is about 2.00x tighter than n=30; n=480 is about 2.00x tighter than n=120).
- Next part uses this same machinery to ask whether a difference between two systems is real or just noise: significance testing.
